In [21]:
%%writefile sample_1.cu
#include <stdio.h>

#define CSC(call)       \
do {                    \
    cudaError_t status = call;          \
    if  (status != cudaSuccess) {       \
        fprintf(stderr, "ERROR in %s:%d. Message: %s\n", __FILE__, __LINE__, cudaGetErrorString(status));   \
        exit(0);                        \
    }                                   \
} while (0)

__global__ void kernel(int *arr, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int offset = blockDim.x * gridDim.x;
    while(idx < n) {
        arr[idx] *= 2;
        idx += offset;
    }
}

int main() {
    int i, n = 100000000;
    int *arr = (int *)malloc(sizeof(int) * n);
    for(i = 0; i < n; i++)
        arr[i] = i;
    int *dev_arr;

    CSC(cudaMalloc(&dev_arr, sizeof(int) * n));
    CSC(cudaMemcpy(dev_arr, arr, sizeof(int) * n, cudaMemcpyHostToDevice));

    cudaEvent_t start, stop;
    CSC(cudaEventCreate(&start));
    CSC(cudaEventCreate(&stop));
    CSC(cudaEventRecord(start));

    kernel<<<512, 512>>>(dev_arr, n);

    CSC(cudaDeviceSynchronize());
    CSC(cudaGetLastError());

    CSC(cudaEventRecord(stop));
    CSC(cudaEventSynchronize(stop));
    float t;
    CSC(cudaEventElapsedTime(&t, start, stop));
    CSC(cudaEventDestroy(start));
    CSC(cudaEventDestroy(stop));

    printf("time = %f ms\n", t);

    CSC(cudaMemcpy(arr, dev_arr, sizeof(int) * n, cudaMemcpyDeviceToHost));
    for(i = 0; i < 10; i++)
        printf("%d ", arr[i]);
    printf("\n");

    free(arr);
    CSC(cudaFree(dev_arr));
    return 0;
}

Overwriting sample_1.cu


In [22]:
%%shell
nvcc sample_1.cu
./a.out

time = 4.160832 ms
0 2 4 6 8 10 12 14 16 18 
